# Predicting Future Values

Now that we have our optimal model, we can use it to predict future contract values for players based on their performance in the most recent 2024-25 NBA season.

In [1]:
import pandas as pd
import numpy as np
import joblib

model = joblib.load('../model/NBA_salary_model.pkl')

In [2]:
# ── Deals data ────────────────────────────────────────────────────────────────
deals_raw = pd.read_csv("../data/NBA_deals_history - deals.csv")
deals_raw.rename(columns={'Age                     At Signing': 'Age'}, inplace=True)
deals_raw.rename(columns={'Team                     Signed With': 'Team'}, inplace=True)
deals_raw['Value'] = deals_raw['Value'].replace('[\\$,]', '', regex=True).astype('Int64')
deals_raw['AAV']   = deals_raw['AAV'].replace('[\\$,]', '', regex=True).astype('Int64')
deals_raw['Age']   = deals_raw['Age'].replace('[\\,]', '', regex=True).astype('Int64')

# ── Stats data ─────────────────────────────────────────────────────────────────
stats_raw = pd.read_csv("../data/kaggle_data/nba_with_2024_25.csv")
stats_raw.drop('TEAM_ID', axis=1, inplace=True)
stats_raw.sort_values(by='PLAYER_ID', ascending=True, inplace=True)
stats_raw['Season_type'] = np.where(
    stats_raw['Season_type'] == 'Regular%20Season', 'RegularSeason', stats_raw['Season_type']
)

wide = stats_raw.pivot(index=['PLAYER_ID', 'PLAYER', 'year'], columns='Season_type').sort_index()
wide.columns = [f'{stat}_{stype}' for stat, stype in wide.columns]
stats = wide.reset_index()
stats.sort_values(by='PLAYER_ID', ascending=True, inplace=True)
stats = stats.drop(columns=['RANK_Playoffs', 'RANK_RegularSeason'])
stats['year'] = stats['year'].str.split('-').str[1].astype(int) + 2000

# ── Rolling averages (stats_roll_final) ────────────────────────────────────────
rolling_stat_map = {
    'PTS': 'PTS_RegularSeason',
    'REB': 'REB_RegularSeason',
    'AST': 'AST_RegularSeason',
    'STL': 'STL_RegularSeason',
    'BLK': 'BLK_RegularSeason',
    'TOV': 'TOV_RegularSeason',
}
windows = [2, 3, 4]

stats_roll = stats[['PLAYER_ID', 'year', 'GP_RegularSeason'] + list(rolling_stat_map.values())].copy()
for stat_name, col in rolling_stat_map.items():
    stats_roll[f'{stat_name}_pg'] = np.where(
        stats_roll['GP_RegularSeason'] > 0,
        stats_roll[col] / stats_roll['GP_RegularSeason'],
        np.nan
    )

stats_roll = stats_roll.sort_values(['PLAYER_ID', 'year']).reset_index(drop=True)
for window in windows:
    for stat_name in rolling_stat_map:
        col_name = f'{stat_name}_mean_{window}yr'
        stats_roll[col_name] = (
            stats_roll.groupby('PLAYER_ID')[f'{stat_name}_pg']
            .transform(lambda x, w=window: x.rolling(w, min_periods=1).mean())
        )

roll_feature_cols = [f'{s}_mean_{w}yr' for s in rolling_stat_map for w in windows]
stats_roll_final = stats_roll[['PLAYER_ID', 'year'] + roll_feature_cols]

# ── Feature constants (mirrors training pipeline) ──────────────────────────────
reg_cats = [
    'MIN_RegularSeason', 'FGM_RegularSeason', 'FGA_RegularSeason', 'FG3M_RegularSeason',
    'FG3A_RegularSeason', 'FTM_RegularSeason', 'FTA_RegularSeason', 'OREB_RegularSeason',
    'DREB_RegularSeason', 'REB_RegularSeason', 'AST_RegularSeason', 'STL_RegularSeason',
    'BLK_RegularSeason', 'TOV_RegularSeason', 'PF_RegularSeason', 'PTS_RegularSeason',
    'EFF_RegularSeason',
]
playoff_cats = [
    'FGM_Playoffs', 'FGA_Playoffs', 'FG3M_Playoffs', 'FG3A_Playoffs', 'FTM_Playoffs',
    'FTA_Playoffs', 'OREB_Playoffs', 'DREB_Playoffs', 'REB_Playoffs', 'AST_Playoffs',
    'STL_Playoffs', 'BLK_Playoffs', 'TOV_Playoffs', 'PF_Playoffs', 'PTS_Playoffs',
    'EFF_Playoffs',
]
cats = [
    'MIN_RegularSeason', 'FGM_Playoffs', 'FGM_RegularSeason', 'FGA_Playoffs', 'FGA_RegularSeason',
    'FG_PCT_Playoffs', 'FG_PCT_RegularSeason', 'FG3M_Playoffs', 'FG3M_RegularSeason', 'FG3A_Playoffs',
    'FG3A_RegularSeason', 'FG3_PCT_Playoffs', 'FG3_PCT_RegularSeason', 'FTM_Playoffs', 'FTM_RegularSeason',
    'FTA_Playoffs', 'FTA_RegularSeason', 'FT_PCT_Playoffs', 'FT_PCT_RegularSeason', 'OREB_Playoffs',
    'OREB_RegularSeason', 'DREB_Playoffs', 'DREB_RegularSeason', 'REB_Playoffs', 'REB_RegularSeason',
    'AST_Playoffs', 'AST_RegularSeason', 'STL_Playoffs', 'STL_RegularSeason', 'BLK_Playoffs',
    'BLK_RegularSeason', 'TOV_Playoffs', 'TOV_RegularSeason', 'PF_Playoffs', 'PF_RegularSeason',
    'PTS_Playoffs', 'PTS_RegularSeason', 'EFF_Playoffs', 'EFF_RegularSeason', 'AST_TOV_Playoffs',
    'AST_TOV_RegularSeason', 'STL_TOV_Playoffs', 'STL_TOV_RegularSeason',
]
cats += [f'{c}_per_game' for c in reg_cats + playoff_cats]
cats += [f'{c}_per36' for c in reg_cats]

counting_stats = [
    'FGM', 'FGA', 'FG3M', 'FG3A', 'FTM', 'FTA',
    'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS', 'EFF', 'MIN',
]
seasons = ['RegularSeason', 'Playoffs']

print(f"Loaded {len(stats)} player-season rows | {stats['year'].nunique()} seasons | deals: {deals_raw.shape[0]} contracts")

Loaded 6837 player-season rows | 13 seasons | deals: 9286 contracts


We will predict values for all players based on updated 2025 data

In [3]:
# Step 1: Extract all 2024-25 players from the full stats history
# year=2025 because of the conversion: "2024-25".split("-")[1] + 2000 = 2025
pred_df = stats[stats['year'] == 2025].copy()

# Players whose teams missed the playoffs get 'MissedPlayoffs' (same as training)
pred_df['TEAM_Playoffs'] = pred_df['TEAM_Playoffs'].fillna('MissedPlayoffs')
pred_df = pred_df.fillna(0)

# Merge rolling averages — computed from the full history (2012-25) so lookbacks
# like PTS_mean_2yr correctly include 2023-24 and 2024-25 for each player
pred_df = pred_df.merge(stats_roll_final, on=['PLAYER_ID', 'year'], how='left')
pred_df = pred_df.fillna(0)

print(f"2024-25 players: {len(pred_df)} total, {int((pred_df['GP_Playoffs'] > 0).sum())} played in playoffs")

2024-25 players: 569 total, 219 played in playoffs


In [4]:
# ── Step 2: Per-game stats (mirrors training pipeline) ───────────────────────
pred_pg = pd.DataFrame(index=pred_df.index)

for stat in reg_cats:
    pred_pg[f'{stat}_per_game'] = np.where(
        pred_df['GP_RegularSeason'] > 0,
        pred_df[stat] / pred_df['GP_RegularSeason'], 0
    )

for stat in playoff_cats:
    pred_pg[f'{stat}_per_game'] = np.where(
        pred_df['GP_Playoffs'] > 0,
        pred_df[stat] / pred_df['GP_Playoffs'], 0
    )

pred_df = pd.concat([pred_df, pred_pg], axis=1)

# ── Step 3: League normalization for 2024-25 ─────────────────────────────────
# Use ALL 2024-25 players for league averages (more representative than just
# contract signers, which is what the training pipeline's groupby('year') uses)
pred_cats_available = [c for c in cats if c in pred_df.columns]
league_2425 = pred_df[pred_cats_available].mean()

for stat in pred_cats_available:
    mean_val = league_2425[stat]
    pred_df[f'{stat}_ratio'] = np.where(mean_val != 0, pred_df[stat] / mean_val - 1, 0)

pred_df = pred_df.fillna(0)
print(f"Per-game and league-normalized ratio features added. Shape: {pred_df.shape}")

Per-game and league-normalized ratio features added. Shape: (569, 178)


In [5]:
# ── Step 4: Additional engineered features (mirrors training pipeline) ────────

# Age and Position: load the full 2024-25 API lookup (all 569 players)
age_pos_lookup = pd.read_csv("../data/player_age_position_2024_25.csv")

# Also pull 2025 contract signers — they have more specific positions (PG/SG/SF/PF)
# and age at signing, which is the correct value for the contract year
deals_2025_info = (
    deals_raw[deals_raw['Start'] == 2025][['Player', 'Age', 'Pos', 'AAV']]
    .rename(columns={'Player': 'PLAYER'})
    .sort_values('AAV', ascending=False)
    .drop_duplicates('PLAYER')
    .drop(columns=['AAV'])
)

# Merge API lookup first (covers everyone), then override with deals data for contract signers
pred_df = pred_df.merge(age_pos_lookup[['PLAYER_ID', 'Age', 'Pos']], on='PLAYER_ID', how='left')

# Override with deals data where available (more specific positions + age at signing)
pred_df = pred_df.merge(deals_2025_info, on='PLAYER', how='left', suffixes=('_api', '_deals'))
pred_df['Age'] = pred_df['Age_deals'].combine_first(pred_df['Age_api']).fillna(0).astype(float)
pred_df['Pos'] = pred_df['Pos_deals'].combine_first(pred_df['Pos_api']).fillna('Unknown')
pred_df = pred_df.drop(columns=['Age_api', 'Age_deals', 'Pos_api', 'Pos_deals'], errors='ignore')

# 1. Age squared
pred_df['Age_squared'] = pred_df['Age'] ** 2

# 2. True Shooting %
denom_rs = 2 * (pred_df['FGA_RegularSeason'] + 0.44 * pred_df['FTA_RegularSeason'])
pred_df['TS_pct_RegularSeason'] = np.where(denom_rs > 0, pred_df['PTS_RegularSeason'] / denom_rs, 0)

denom_po = 2 * (pred_df['FGA_Playoffs'] + 0.44 * pred_df['FTA_Playoffs'])
pred_df['TS_pct_Playoffs'] = np.where(
    pred_df['GP_Playoffs'] > 0,
    np.where(denom_po > 0, pred_df['PTS_Playoffs'] / denom_po, 0),
    pred_df['TS_pct_RegularSeason']
)

# 3. Usage rate proxy (per 36 min)
pred_df['USG_proxy_RegularSeason'] = np.where(
    pred_df['MIN_RegularSeason'] > 0,
    (pred_df['FGA_RegularSeason'] + 0.44 * pred_df['FTA_RegularSeason'] + pred_df['TOV_RegularSeason']) / pred_df['MIN_RegularSeason'] * 36, 0
)
pred_df['USG_proxy_Playoffs'] = np.where(
    pred_df['GP_Playoffs'] > 0,
    np.where(pred_df['MIN_Playoffs'] > 0,
        (pred_df['FGA_Playoffs'] + 0.44 * pred_df['FTA_Playoffs'] + pred_df['TOV_Playoffs']) / pred_df['MIN_Playoffs'] * 36, 0),
    pred_df['USG_proxy_RegularSeason']
)

# 4. Box composite (PTS + REB + AST per game)
pred_df['BoxComposite_RegularSeason'] = (
    pred_df['PTS_RegularSeason_per_game'] + pred_df['REB_RegularSeason_per_game'] + pred_df['AST_RegularSeason_per_game']
)
pred_df['BoxComposite_Playoffs'] = np.where(
    pred_df['GP_Playoffs'] > 0,
    pred_df['PTS_Playoffs_per_game'] + pred_df['REB_Playoffs_per_game'] + pred_df['AST_Playoffs_per_game'],
    pred_df['BoxComposite_RegularSeason']
)

# 5. Playoff uplift (0 for non-playoff players — no penalty)
pred_df['PTS_uplift'] = np.where(pred_df['GP_Playoffs'] > 0, pred_df['PTS_Playoffs_per_game'] - pred_df['PTS_RegularSeason_per_game'], 0)
pred_df['REB_uplift'] = np.where(pred_df['GP_Playoffs'] > 0, pred_df['REB_Playoffs_per_game'] - pred_df['REB_RegularSeason_per_game'], 0)
pred_df['AST_uplift'] = np.where(pred_df['GP_Playoffs'] > 0, pred_df['AST_Playoffs_per_game'] - pred_df['AST_RegularSeason_per_game'], 0)

# 6. Made playoffs flag
pred_df['made_playoffs'] = (pred_df['GP_Playoffs'] > 0).astype(int)

pred_df = pred_df.fillna(0)
print(f"Additional features added.")
print(f"  Players with known Age: {(pred_df['Age'] > 0).sum()} / {len(pred_df)}")
print(f"  Players with known Pos: {(pred_df['Pos'] != 'Unknown').sum()} / {len(pred_df)}")
print(f"  Age range: {pred_df['Age'].min():.0f} – {pred_df['Age'].max():.0f}")
# Spot-check: LeBron should have age ~40
lebron = pred_df[pred_df['PLAYER'] == 'LeBron James']
if len(lebron):
    print(f"  LeBron James → Age={lebron['Age'].values[0]}, Pos={lebron['Pos'].values[0]}")

Additional features added.
  Players with known Age: 569 / 569
  Players with known Pos: 569 / 569
  Age range: 19 – 40
  LeBron James → Age=40.0, Pos=F


In [6]:
# ── Step 5: Build X_2425 and predict ─────────────────────────────────────────
cap_2025_26 = 154_647_000  # 2025-26 NBA salary cap

# One-hot encode categoricals (same as training)
pred_X = pd.get_dummies(pred_df, columns=['Pos', 'TEAM_Playoffs', 'TEAM_RegularSeason'], drop_first=True)
pred_X = pred_X.astype({col: 'Int64' for col in pred_X.select_dtypes('bool').columns})

# Drop per-36 and raw counting totals (same reduction as training X)
per36_pred   = pred_X.filter(regex='_per36').columns.tolist()
raw_tot_pred = [c for c in pred_X.columns if any(c in (f'{s}_{t}', f'{s}_{t}_ratio') for s in counting_stats for t in seasons)]
pred_X = pred_X.drop(columns=list(set(per36_pred + raw_tot_pred)), errors='ignore')

# Save player identifiers before aligning to model features
player_info = pred_df[['PLAYER', 'PLAYER_ID']].copy()

# Align to model's exact feature set — new dummies get 0, unseen dummies are dropped
pred_X_model = pred_X.drop(columns=['PLAYER', 'PLAYER_ID', 'year'], errors='ignore')
pred_X_model = pred_X_model.reindex(columns=model.feature_names_in_, fill_value=0)

print(f"Prediction matrix: {pred_X_model.shape[0]} players × {pred_X_model.shape[1]} features")

Prediction matrix: 569 players × 261 features


In [7]:
# Predict
pred_ratios = model.predict(pred_X_model)

# Build results table
results_2425 = player_info.copy()
results_2425['Predicted_SalaryRatio'] = pred_ratios.round(4)
results_2425['Predicted_AAV']         = (pred_ratios * cap_2025_26).round(0).astype(int)

results_2425 = results_2425.sort_values('Predicted_AAV', ascending=False).reset_index(drop=True)
results_2425.head(30)

,PLAYER,PLAYER_ID,Predicted_SalaryRatio,Predicted_AAV
0,Shai Gilgeous-Alexander,1628983,0.3635,56212455
1,Luka Dončić,1629029,0.3620,55977432
2,Nikola Jokić,203999,0.3606,55758813
3,Giannis Antetokounmpo,203507,0.3592,55550377
4,LeBron James,2544,0.3566,55146392
5,Jayson Tatum,1628369,0.3505,54197224
6,Jalen Brunson,1628973,0.3474,53718889
7,Stephen Curry,201939,0.3450,53356770
8,Anthony Edwards,1630162,0.3397,52536129
9,Donovan Mitchell,1628378,0.3338,51617024
